In [1]:
# 1_data_preprocessing.ipynb
"""
Customer Churn Analysis - Data Preprocessing
Task 1: Load and preprocess the dataset, addressing missing values,
and encoding categorical variables for machine learning readiness.
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TASK 1: DATA PREPROCESSING")
print("Customer Churn Analysis - Telecommunications Dataset")
print("="*80)

# Create necessary directories
os.makedirs('data/processed', exist_ok=True)

# Step 1: Load the dataset
print("\n📂 STEP 1: LOADING DATASET")
print("-"*50)

try:
    df = pd.read_csv('data/raw/Telco_Customer_Churn_Dataset .csv')
    print(f"✅ Dataset loaded successfully!")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
except:
    try:
        df = pd.read_csv('Telco_Customer_Churn_Dataset.csv')
        print(f"✅ Dataset loaded from current directory!")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    except:
        print("❌ Error: Dataset not found!")
        print("Please place Telco_Customer_Churn_Dataset.csv in the data/raw/ folder")
        exit()

print(f"\n📊 Dataset Overview:")
print(f"   Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

# Step 2: Check for missing values
print("\n🔍 STEP 2: CHECKING MISSING VALUES")
print("-"*50)

missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_report = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
print(missing_report[missing_report['Missing Count'] > 0])

if missing_values.sum() == 0:
    print("✅ No missing values found in the dataset!")
else:
    print(f"⚠️ Found {missing_values.sum()} missing values in the dataset")

# Step 3: Handle missing values
print("\n🛠️ STEP 3: HANDLING MISSING VALUES")
print("-"*50)

# Convert TotalCharges to numeric (it's stored as string)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("✅ Converted 'TotalCharges' from string to numeric")

# Drop rows with missing TotalCharges (very few)
initial_rows = len(df)
df = df.dropna(subset=['TotalCharges'])
rows_dropped = initial_rows - len(df)
print(f"✅ Dropped {rows_dropped} rows with missing TotalCharges")

# Drop customerID (not useful for prediction)
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)
    print("✅ Dropped 'customerID' column (not useful for prediction)")

print(f"\n📊 Dataset after handling missing values:")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Missing values remaining: {df.isnull().sum().sum()}")

# Step 4: Encode categorical variables
print("\n🔤 STEP 4: ENCODING CATEGORICAL VARIABLES")
print("-"*50)

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Encode target variable (Yes=1, No=0)
y_encoded = y.map({'Yes': 1, 'No': 0})
print(f"✅ Target variable encoded: Yes → 1, No → 0")
print(f"   Distribution:\n{y_encoded.value_counts()}")

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\n📋 Column Types:")
print(f"   Numerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"   Categorical columns ({len(categorical_cols)}): {categorical_cols}")

# One-hot encoding for categorical variables
print("\n🔄 Performing one-hot encoding...")
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"✅ Encoding complete!")
print(f"   Original features: {X.shape[1]}")
print(f"   Encoded features: {X_encoded.shape[1]}")

# Step 5: Feature Engineering (Advanced - Makes project stand out)
print("\n⚙️ STEP 5: FEATURE ENGINEERING (ADVANCED)")
print("-"*50)

# Create tenure groups (helps capture non-linear relationships)
if 'tenure' in X_encoded.columns:
    X_encoded['tenure_group'] = pd.cut(X_encoded['tenure'], 
                                       bins=[0, 12, 24, 48, 72], 
                                       labels=['new', 'moderate', 'loyal', 'veteran'])
    tenure_dummies = pd.get_dummies(X_encoded['tenure_group'], prefix='tenure')
    X_encoded = pd.concat([X_encoded, tenure_dummies], axis=1)
    X_encoded = X_encoded.drop('tenure_group', axis=1)
    print("✅ Added tenure group features (new, moderate, loyal, veteran)")

# Create charges per tenure ratio (financial commitment indicator)
if 'MonthlyCharges' in X_encoded.columns and 'tenure' in X_encoded.columns:
    X_encoded['charges_per_tenure'] = X_encoded['MonthlyCharges'] / (X_encoded['tenure'] + 1)
    print("✅ Added 'charges_per_tenure' feature")

# Create high-risk indicator (month-to-month + high charges)
if 'Contract_Month-to-month' in X_encoded.columns and 'MonthlyCharges' in X_encoded.columns:
    X_encoded['high_risk'] = ((X_encoded['Contract_Month-to-month'] == 1) & 
                              (X_encoded['MonthlyCharges'] > 70)).astype(int)
    print("✅ Added 'high_risk' indicator feature")

print(f"\n📊 Final feature count: {X_encoded.shape[1]}")

# Step 6: Save preprocessed data
print("\n💾 STEP 6: SAVING PREPROCESSED DATA")
print("-"*50)

X_encoded.to_csv('data/processed/X_processed.csv', index=False)
y_encoded.to_csv('data/processed/y_processed.csv', index=False)

print(f"✅ Features saved to: data/processed/X_processed.csv")
print(f"✅ Target saved to: data/processed/y_processed.csv")

# Step 7: Data statistics and summary
print("\n📊 STEP 7: DATA STATISTICS SUMMARY")
print("-"*50)

print(f"\n🎯 Target Variable Distribution:")
churn_count = y_encoded.sum()
no_churn_count = len(y_encoded) - churn_count
churn_percentage = (churn_count / len(y_encoded)) * 100
print(f"   Churn (Yes): {churn_count} ({churn_percentage:.1f}%)")
print(f"   No Churn (No): {no_churn_count} ({100 - churn_percentage:.1f}%)")

print(f"\n📈 Numerical Features Statistics:")
print(X_encoded.describe().round(2))

print("\n" + "="*80)
print("✅ TASK 1 COMPLETED: DATA PREPROCESSING")
print("="*80)
print("\n📝 Summary of Actions:")
print("1. ✅ Loaded Telco Customer Churn dataset")
print("2. ✅ Handled missing values in TotalCharges")
print("3. ✅ Removed customerID (not useful for ML)")
print("4. ✅ Encoded target variable (Yes=1, No=0)")
print("5. ✅ Performed one-hot encoding for categorical variables")
print("6. ✅ Added advanced features (tenure groups, risk indicators)")
print("7. ✅ Saved preprocessed data for model training")
print("\n🚀 Ready for next step: Data Splitting (Task 2)")

TASK 1: DATA PREPROCESSING
Customer Churn Analysis - Telecommunications Dataset

📂 STEP 1: LOADING DATASET
--------------------------------------------------
✅ Dataset loaded successfully!
   Shape: 7043 rows × 21 columns

📊 Dataset Overview:
   Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  923